# Tool Calling

In [1]:
from agents import Agent, ModelSettings, Runner, function_tool, trace
from setup import bedrock_tool

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


Create a static calorie table that we can use as a tool:

In [2]:
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

Let's test this out: 

In [3]:
get_food_calories('banana')

'Banana: 105 calories per medium banana (118g)'

In [4]:
@function_tool
def get_food_calories_tool(food_item: str) -> str:
    return get_food_calories(food_item)

In [5]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(get_food_calories_tool.__dict__)],
)

In [6]:
with trace("Nutrition Assistant with tools") as t:
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: 
Banana: 105 calories
Apple: 80 calories
Total: 185 calories



The total calories in a banana and an apple are 185 calories.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_0708427182f8419f849c9e2f4c15a8f0
